In [50]:
from datasets import load_dataset

dataset = load_dataset("wandb/RAGTruth-processed")

Repo card metadata block was not found. Setting CardData to empty.


In [51]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 15090
    })
    test: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 2700
    })
})


In [52]:
import numpy as np
import pandas as pd

# Convert the 'train' split of the dataset to a pandas DataFrame
df = pd.DataFrame(dataset["train"])

In [53]:
def to_label(processed: dict[str, int]) -> int:
    return 0 if all(v == 0 for v in processed.values()) else 1


df["label"] = df["hallucination_labels_processed"].apply(to_label)

In [54]:
df["context"].nunique()

2514

If I split rows randomly, the same article ends up in train and val. The model can score well by remembering facts about articles it already saw (B) instead of learning to check facts in general (A). The val score is too optimistic, and it will drop on new documents like DocMind’s.

In [55]:
test_df = pd.DataFrame(dataset["test"])
test_df["context"].nunique()

450

In [56]:
train_contexts = set(df["context"])
test_contexts = set(test_df["context"])

In [57]:
example = df["context"].value_counts().index[0]
df[df["context"] == example][["model", "task_type", "label", "output"]]

,model,task_type,label,output
3102,gpt-4-0613,Summary,0,The National Weather Service has predicted tor...
3103,gpt-3.5-turbo-0613,Summary,0,The National Weather Service warns of tornadoe...
3104,mistral-7B-instruct,Summary,1,"Tornadoes, strong winds and thunderstorms with..."
3105,llama-2-7b-chat,Summary,0,Here is a summary of the news in 42 words or l...
3106,llama-2-13b-chat,Summary,0,Sure! Here's the summary within 42 words:\n\nS...
3107,llama-2-70b-chat,Summary,0,The National Weather Service predicts tornadoe...
3108,gpt-4-0613,Summary,1,The Midwest and Plains are expected to see tor...
3109,gpt-3.5-turbo-0613,Summary,0,The Midwest and Plains are expected to experie...
3110,mistral-7B-instruct,Summary,1,"Tornadoes, strong winds and thunderstorms with..."
3111,llama-2-7b-chat,Summary,0,Here is a summary of the news in 42 words or l...


In [58]:
def shared_contexts(a: pd.DataFrame, b: pd.DataFrame) -> set[str]:
    return set(a["context"]) & set(b["context"])


shared_contexts(df, test_df)

set()

In [59]:
def split_by_context(
    df: pd.DataFrame, val_frac: float = 0.15, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split df into (train, val) so that no context is in both."""
    ctx = df["context"].unique()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(ctx)
    n = int(len(shuffled) * val_frac)
    val_ctx = shuffled[:n]
    mask = df["context"].isin(val_ctx)
    val = df[mask].copy()
    train = df[~mask].copy()
    return train, val

In [60]:
train, val = split_by_context(df)

print(len(train), len(val))
print(len(shared_contexts(train, val)))
print(train["label"].mean(), val["label"].mean())

12828 2262
0
0.44808231992516373 0.4301503094606543


In [61]:
print(train["task_type"].value_counts(normalize=True))
print(val["task_type"].value_counts(normalize=True))

task_type
Data2txt    0.353601
QA          0.331618
Summary     0.314780
Name: proportion, dtype: float64
task_type
QA          0.344828
Data2txt    0.336870
Summary     0.318302
Name: proportion, dtype: float64


In [62]:
train.groupby("context")["task_type"].nunique()

context
\n{'name': "A Slice of Woodstock's", 'address': 'UCSB University Center', 'city': 'Santa Barbara', 'state': 'CA', 'categories': 'Pizza, Gluten-Free, Restaurants, Event Planning & Services, Caterers', 'hours': {'Monday': '0:0-0:0', 'Tuesday': '10:30-15:30', 'Wednesday': '10:30-15:30', 'Thursday': '10:30-15:30', 'Friday': '10:30-15:30'}, 'attributes': {'BusinessParking': None, 'RestaurantsReservations': False, 'OutdoorSeating': True, 'WiFi': 'no', 'RestaurantsTakeOut': True, 'RestaurantsGoodForGroups': True, 'Music': None, 'Ambience': None}, 'business_stars': 4.0, 'review_info': [{'review_stars': 1.0, 'review_date': '2021-09-30 14:43:47', 'review_text': "Got out of lab hella hungry. Wanted to see if I could get some pizza. Google said they were open. The person there just ignored me so I'm not going there again."}, {'review_stars': 5.0, 'review_date': '2019-08-19 01:59:01', 'review_text': "A slice of Woodstock's is my favorite on campus spot to grab a quick lunch. Fast service, n

In [63]:
counts = df.groupby("context")["task_type"].nunique()
mask = counts > 1
print(mask.sum())

0


In [64]:
def split_stratified(
    df: pd.DataFrame, val_frac: float = 0.15, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_list = []
    val_list = []
    for task in df["task_type"].unique():
        mask = df["task_type"] == task
        task_train, val_train = split_by_context(df[mask], val_frac=val_frac, seed=seed)
        train_list.append(task_train)
        val_list.append(val_train)
    train = pd.concat(train_list, ignore_index=True)
    val = pd.concat(val_list, ignore_index=True)
    return train, val

In [74]:
train, val = split_stratified(df)

print(len(train), len(val))
print(len(shared_contexts(train, val)))
print(train["label"].mean(), val["label"].mean())

12834 2256
0
0.44530154277699857 0.44592198581560283


In [66]:
df["output_words"] = df["output"].str.split().str.len()
df["output_words"].describe()

count    15090.000000
mean       131.363685
std         57.214785
min          1.000000
25%         92.000000
50%        125.000000
75%        163.000000
max        632.000000
Name: output_words, dtype: float64

In [67]:
df["output_words"] = df["output"].str.split().str.len()

answers = df.sort_values("output_words").head(10)[
    ["output_words", "output", "label", "task_type", "model"]
]
answers

,output_words,output,label,task_type,model
12397,1,exclude,0,QA,gpt-3.5-turbo-0613
13652,5,Pasta is feminine in Italian.,1,QA,mistral-7B-instruct
10850,5,types of food in france,0,QA,mistral-7B-instruct
11947,7,Unable to answer based on given passages.,0,QA,gpt-3.5-turbo-0613
10117,7,Unable to answer based on given passages.,0,QA,gpt-3.5-turbo-0613
10124,7,Unable to answer based on given passages.,0,QA,mistral-7B-instruct
13836,7,Unable to answer based on given passages.,0,QA,gpt-4-0613
11840,7,Unable to answer based on given passages.,0,QA,mistral-7B-instruct
13741,7,Unable to answer based on given passages.,0,QA,gpt-3.5-turbo-0613
13908,7,Unable to answer based on given passages.,0,QA,gpt-4-0613


In [68]:
empty = df["output"].str.strip() == ""  # True/False for every row
print(empty.sum())

0


In [69]:
mask = counts > 1
mask = df["context"] != df["context"].str.strip()
print(mask.sum())

5304


In [70]:
print(df["context"].str.strip().nunique())

2514


In [71]:
mask = df["output"] != df["output"].str.strip()
print(mask.sum())

0
